In [2]:
from dataclasses import dataclass
import torch
from torch.nn import functional as F
import torch.nn as nn
import tiktoken
import requests

In [3]:
class GPTConfig:
    context_length = 32 ### Specifying the Context Length which the Transformer will be available to Process
    vocab_size = 50257 ### Vocab Size -- Tiktoken Vocab Size -- 50,257 also include <|endoftext|>, print(tiktoken.get_encoding('gpt2').n_vocab)
    n_layer = 4 ### No of the Transformer Decoder stacks
    n_head = 4 ### No of Attention Heads Used
    n_embd = 128 ### Dimensionality of the Word Embedding 

In [4]:
class DecoderBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.layer_norm1 = nn.LayerNorm(config.n_embd)
        self.attn = nn.MultiheadAttention(embed_dim=config.n_embd, num_heads=config.n_head, batch_first=True, dropout=0.2)
        self.layer_norm2 = nn.LayerNorm(config.n_embd)
        self.multi_layer_perceptron = nn.Sequential(
            nn.Linear(config.n_embd, 4 * config.n_embd),
            nn.GELU(),
            nn.Linear(4 * config.n_embd, config.n_embd)
        )
    def forward(self, x):
        B, T, C = x.shape ## Getting the Batch, time and Channel dimension from the input
        mask = torch.triu(torch.ones(T, T), diagonal=1).bool() ## Adding the Mask -- Disabling the Upper Triangle.
        attn_out, _ = self.attn(self.layer_norm1(x), self.layer_norm1(x), self.layer_norm1(x), attn_mask=mask) ## It returns Attention Weights too
        x = x + attn_out ## Applied Residual Masked Multihead Attention with Causality

        x = x + self.multi_layer_perceptron(self.layer_norm2(x))
        return x

In [5]:
class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(
            token_embedding = nn.Embedding(config.vocab_size, config.n_embd),
            pos_embedding = nn.Embedding(config.context_length, config.n_embd),
            decoder = nn.ModuleList([DecoderBlock(config) for _ in range(config.n_layer)]),
            layer_norm = nn.LayerNorm(config.n_embd)
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.transformer.token_embedding(idx)
        pos = torch.arange(0, T)
        pos_emb = self.transformer.pos_embedding(pos)

        x = tok_emb + pos_emb
        for Block in self.transformer.decoder:
            x = Block(x) ### Passing through all 4 Decoder Blocks
        
        x = self.transformer.layer_norm(x)
        logits = self.lm_head(x) ### B, T, n_embd ----> B, T, Vocab_Size

        loss = None

        if targets is not None:
            B, T, V = logits.shape
            logits = logits.view(B*T, V)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

In [6]:
model = GPT(GPTConfig())
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

In [7]:
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

In [8]:
total_params

13713489